# Transmission through absorbing layers

Combine three synthetic wavelength-dependent absorbers. This is a scalar
Beer-Lambert model for independent absorption: it neglects reflections,
interference, scattering, saturation, and fluorescence. The spectra below
are illustrative inputs, not measured material data.
Run the setup in [examples/README.md](README.md) first.

If layer $j$ has thickness $d_j$ and an intensity absorption coefficient
$\alpha_j(\lambda)$ with inverse-length units, then
$$
T_j(\lambda)=e^{-\alpha_j(\lambda)d_j},\qquad
T_{\rm stack}(\lambda)=\prod_j T_j(\lambda),\qquad
I_{\rm out}=I_{\rm in}T_{\rm stack}.
$$
This $\alpha$ uses a natural exponential and is not a base-10 molar
absorptivity. $I$ is monochromatic irradiance at the scanned wavelength.

Reference: [Duke CHEM 401L, Absorbance and Beer's Law](https://chem.libretexts.org/Courses/Duke_University/CHEM_401L%3A_Analytical_Chemistry_Lab/06%3A_Instrument_Facilities_for_CHEM401L/02%3A_Absorption_Spectrometers_%28UV-vis%29/2.01%3A_Absorbance_and_Beer%27s_Law).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx import Quantity
from unitx.si import si

wavelength = np.linspace(400, 800, 401) * si.nm
centers = np.asarray([450, 560, 680])[:, None] * si.nm
widths = np.asarray([25, 35, 30])[:, None] * si.nm
baseline = np.asarray([200, 250, 150])[:, None] / si.cm
peak = np.asarray([2000, 4000, 3000])[:, None] / si.cm
thickness = np.asarray([10, 15, 20])[:, None] * si.um
alpha = baseline + peak * np.exp(-(((wavelength - centers) / widths) ** 2) / 2)
layer_transmission = np.exp(-alpha * thickness)
assert isinstance(layer_transmission, Quantity)

## Multiply transmissions and select active layers

`axis=0` reduces the layer axis. A boolean input mask can omit the middle
layer without pretending that omitted values are zero transmission. Since
each transmission is dimensionless, a zero-layer stack would transmit one.

In [ ]:
full_stack = np.prod(layer_transmission, axis=0)
active_layers = np.array([True, False, True])[:, None]
selected_stack = np.multiply.reduce(layer_transmission, axis=0, where=active_layers)
assert isinstance(full_stack, Quantity)
assert isinstance(selected_stack, Quantity)
incident = 1000.0 * si.W / si.m**2
transmitted = np.empty(wavelength.size) * si.mW / si.cm**2
# NumPy's stubs do not describe Quantity initial/out protocol dispatch.
result = np.prod(  # pyright: ignore[reportCallIssue]
    layer_transmission,
    axis=0,
    initial=incident,  # pyright: ignore[reportArgumentType]
    out=transmitted,  # pyright: ignore[reportArgumentType]
)
assert result is transmitted
print(
    f'Output irradiance range [{transmitted.unit}]:',
    transmitted.value.min(),
    transmitted.value.max(),
)

## Read the spectra

The second panel uses the incident irradiance as the product's `initial`.
Its unit contributes once; `out` converts to a fixed target unit and keeps
that target's ndarray and dtype.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
axes[0].plot(wavelength.value, layer_transmission.value.T, alpha=0.65)
axes[0].plot(wavelength.value, full_stack.value, 'k', lw=2, label='All layers')
axes[0].plot(wavelength.value, selected_stack.value, '--', label='Layers 1 + 3')
axes[0].set(
    xlabel=f'Wavelength [{wavelength.unit}]',
    ylabel='Transmission',
    ylim=(0, 1.02),
    title='Independent absorption',
)
axes[0].legend()
axes[1].plot(wavelength.value, transmitted.value)
axes[1].set(
    xlabel=f'Wavelength [{wavelength.unit}]',
    ylabel=f'Irradiance [{transmitted.unit}]',
    title=f'Monochromatic input: {incident}',
)
plt.show()

## Check optical depths and a dimensional product

Products of exponentials must agree with the exponential of the summed
optical depth. A separate geometric product illustrates $U^N$: stacking
three lengths into one unit and multiplying gives volume, not length.

In [ ]:
assert np.all(full_stack.isclose(np.exp(-np.sum(alpha * thickness, axis=0))))
assert np.all(transmitted.isclose(incident * full_stack))
assert np.all(selected_stack >= full_stack)
assert np.all((full_stack.value >= 0) & (full_stack.value <= 1))
dimensions = np.stack((20 * si.mm, 10 * si.mm, np.sum(thickness)))
active_volume = np.prod(dimensions)
assert isinstance(active_volume, Quantity)
assert active_volume.isclose(9 * si.mm**3)
print('Absorbing-material volume:', active_volume)